In [1]:
import numpy as np
from collections import Counter
from itertools import combinations, permutations
from ase.build import surface, make_supercell
from ase.io import write, read
from ase.visualize import view
from ase import Atom
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.io.lammps.data import LammpsData
from pymatgen.core.surface import Slab
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import os

os.makedirs("slabs", exist_ok=True)

ld = LammpsData.from_file('bulk_structure/mg2zn11_relaxed.data', atom_style='atomic')
alloy = AseAtomsAdaptor().get_atoms(ld.structure)
print(f"Bulk: {len(alloy)} atoms, {alloy.get_chemical_formula()}")

Bulk: 39 atoms, Mg6Zn33


In [2]:
slab = surface(alloy, (1, 1, 1), 6, vacuum=8)
sc = make_supercell(slab, [[2,0,0],[0,2,0],[0,0,1]])

sym = np.array(sc.get_chemical_symbols())
mg, zn = sum(sym == 'Mg'), sum(sym == 'Zn')
z = sc.get_positions()[:, 2]
thick = z.max() - z.min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
slab_obj = Slab(ps.lattice, ps.species, ps.frac_coords,
    miller_index=(1,1,1), oriented_unit_cell=ps, shift=0,
    scale_factor=[[1,0,0],[0,1,0],[0,0,1]])

print(f"Atoms: {len(sc)}, Mg: {mg}, Zn: {zn}")
print(f"Thickness: {thick:.1f} A")
print(f"Area: {area:.1f} A²")
print(f"Stoichiometric: {'YES' if mg*11 == zn*2 else 'NO'}")
print(f"Symmetric: {slab_obj.is_symmetric()}")

Atoms: 936, Mg: 144, Zn: 792
Thickness: 29.3 A
Area: 525.7 A²
Stoichiometric: YES
Symmetric: True


In [3]:
view(sc)

<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [4]:
thick = sc.get_positions()[:,2].max() - sc.get_positions()[:,2].min()
area = np.linalg.norm(np.cross(sc.cell[0], sc.cell[1]))

ps = AseAtomsAdaptor().get_structure(sc)
ld = LammpsData.from_structure(ps, atom_style="atomic")
ld.write_file("slabs/mg2zn11_111_6layers_reconstructed_ase.data")

print(f"Saved: slabs/mg2zn11_111_6layers_reconstructed_ase.data")
print(f"  Atoms: {len(sc)}")
print(f"  Thickness: {thick:.1f} A")
print(f"  Area: {area:.1f} A²")
print(f"  Stoichiometric: YES")
print(f"  Symmetric: TRUE")

view(sc)

Saved: slabs/mg2zn11_111_6layers_reconstructed_ase.data
  Atoms: 936
  Thickness: 29.3 A
  Area: 525.7 A²
  Stoichiometric: YES
  Symmetric: TRUE


<Popen: returncode: None args: ['C:\\Users\\Kai Savage\\anaconda3\\python.ex...>

In [5]:
#unrelaxed surface energy calculation
E_slab = -1015.0208       # eV
E_bulk_per_fu =  -44.119792 / 3  # eV per formula unit 
n = 936 / 13                 # formula units in slab = 32
A = 525.7             # Ang^2

S_eV_per_ang2 = (E_slab - n * E_bulk_per_fu) / (2 * A)

# Convert to J/m^2 (1 eV/Ang^2 = 16.0218 J/m^2)
S_J_per_m2 = S_eV_per_ang2 * 16.0218

print(f"E_bulk per formula unit = {E_bulk_per_fu:.6f} eV")
print(f"n * E_bulk = {n * E_bulk_per_fu:.5f} eV")
print(f"E_slab - n*E_bulk = {E_slab - n * E_bulk_per_fu:.5f} eV")
print(f"S = {S_eV_per_ang2:.6f} eV/Ang^2")
print(f"S = {S_J_per_m2:.4f} J/m^2")

E_bulk per formula unit = -14.706597 eV
n * E_bulk = -1058.87501 eV
E_slab - n*E_bulk = 43.85421 eV
S = 0.041710 eV/Ang^2
S = 0.6683 J/m^2


In [6]:
#relaxed surface energy calculation
E_slab_relaxed =  -1022.16909626602    # eV
E_bulk_per_fu = -44.119792 / 3  # eV per formula unit
n = 936 / 13                      # 32 formula units
A = 525.7              # Ang^2

S_relaxed_eV = (E_slab_relaxed - n * E_bulk_per_fu) / (2 * A)
S_relaxed_Jm2 = S_relaxed_eV * 16.0218

print(f"E_slab_relaxed - n*E_bulk = {E_slab_relaxed - n * E_bulk_per_fu:.5f} eV")
print(f"S relaxed = {S_relaxed_eV:.6f} eV/Ang^2")
print(f"S relaxed = {S_relaxed_Jm2:.4f} J/m^2")

# Compare with unrelaxed
S_unrelaxed_Jm2 =  0.6683
print(f"\nUnrelaxed S = {S_unrelaxed_Jm2:.4f} J/m^2")
print(f"Relaxed S   = {S_relaxed_Jm2:.4f} J/m^2")
print(f"Relaxation energy = {S_unrelaxed_Jm2 - S_relaxed_Jm2:.4f} J/m^2")

E_slab_relaxed - n*E_bulk = 36.70591 eV
S relaxed = 0.034911 eV/Ang^2
S relaxed = 0.5593 J/m^2

Unrelaxed S = 0.6683 J/m^2
Relaxed S   = 0.5593 J/m^2
Relaxation energy = 0.1090 J/m^2
